# 01_data_exploration

Load, inspect and visualise the Pima Indians Diabetes dataset.

---

*DiaPredict AI — Explainable Diabetes Risk Prediction System.* *Educational / research prototype, not a medical diagnosis.*

## Objectives

In this notebook we explore the Pima Indians Diabetes dataset:
- Load the raw data
- Understand the schema and summary statistics
- Visualise feature distributions and class balance
- Motivate the preprocessing decisions used later (missing-value handling).

In [ ]:
import sys
from pathlib import Path

# Make the project root importable no matter where the notebook runs.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "backend").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
print("Root:", PROJECT_ROOT)


### 1. Load the dataset

In [ ]:
from backend.app.preprocessing import load_raw_dataset
from backend.app.config import FEATURES, TARGET

df = load_raw_dataset()
print(f"Samples: {len(df)}, features: {len(FEATURES)}")
df.head()

### 2. Summary statistics

In [ ]:
df.describe().T


> Notice that Glucose, BloodPressure, SkinThickness, Insulin and BMI have a minimum of `0`, which is physiologically impossible. In the original Pima dataset, `0` is used as a placeholder for a missing measurement. We handle this in preprocessing.


### 3. Class balance (target)

In [ ]:
print(df[TARGET].value_counts())
df[TARGET].value_counts(normalize=True).plot(kind="bar", color=["#60a5fa", "#f87171"])
plt.title("Outcome distribution (0 = no diabetes, 1 = diabetes)")
plt.ylabel("Proportion")
plt.show()

### 4. Feature distributions by class

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, col in zip(axes.ravel(), FEATURES):
    sns.histplot(data=df, x=col, hue=TARGET, kde=True, ax=ax, palette=["#60a5fa", "#f87171"])
    ax.set_title(col)
for ax in axes.ravel()[len(FEATURES):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 5. Correlation heat map

In [ ]:
plt.figure(figsize=(11, 9))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True, cbar_kws={"shrink": .8})
plt.title("Feature correlation matrix")
plt.show()

### 6. Observations

- `Glucose` has the strongest single correlation with the outcome.
- The dataset is **imbalanced** (roughly 35% positive), which is why evaluating precision / recall / ROC-AUC matters more than raw accuracy.
- Several columns contain placeholder zeros that must be treated as missing values.